In [16]:
from pathlib import Path
import shutil

In [ ]:
def infer_base_label(folder_name: str) -> str | None:
    """
    Infer task label from raw experiment folder name.

    Examples
    --------
    241213 first-pattern      -> first-pattern
    241214 pattern            -> pattern
    241224 position           -> position
    250101 couple             -> couple
    250101 pure couple        -> couple

    250101 75 pattern         -> pattern
    250101 75 position        -> position
    250101 75 couple          -> couple
    """
    name = folder_name.lower()
    name = name.replace("_", " ")
    name = name.replace("-", " ")
    name = name.replace("%", " ")
    name = " ".join(name.split())

    has_first = "first" in name

    has_couple = (
        "couple" in name
        or "coupled" in name
        or "pair" in name
    )

    has_pattern = (
        "pattern" in name
        or "pat" in name
    )

    has_position = (
        "position" in name
        or "pos" in name
    )

    # First-session folders have higher priority.
    # Example:
    #   "75 first pattern" -> first-pattern
    # even if the data internally contains couple_* tracks.
    if has_first and has_pattern:
        return "first-pattern"

    if has_first and has_position:
        return "first-position"

    # Pure couple session.
    if has_couple:
        return "couple"

    if has_pattern:
        return "pattern"

    if has_position:
        return "position"

    return None


def infer_reward_mode(folder_name: str) -> str:
    """
    Infer reward schedule from raw experiment folder name.

    Returns
    -------
    reward_mode : str
        "75"
            Correct trials are rewarded with 75% probability,
            and NoReward behavior type may exist.

        "full"
            Standard fully rewarded correct trials.

    Supported examples
    ------------------
    250101 75 pattern       -> 75
    250101 75-position      -> 75
    250101 75% couple       -> 75
    250101 pattern          -> full
    """
    name = folder_name.lower()
    name = name.replace("_", " ")
    name = name.replace("-", " ")
    name = name.replace("%", " ")
    name = " ".join(name.split())

    tokens = name.split()

    # Strict token-level detection to avoid matching dates such as 2025.
    if "75" in tokens:
        return "75"

    return "full"


def infer_experiment_labels(folder_name: str) -> dict | None:
    """
    Infer both task label and reward mode from one experiment folder.

    Returns
    -------
    dict or None
        {
            "base_label": "pattern" / "position" / "couple" / ...,
            "reward_mode": "full" / "75",
            "file_label": "pattern" / "75_pattern" / ...
        }

    Naming convention
    -----------------
    For full reward:
        file_label = base_label

    For 75% reward:
        file_label = 75_<base_label>

    Examples
    --------
    241214 pattern       -> pattern
    250101 75 pattern    -> 75_pattern
    250101 75 couple     -> 75_couple
    """
    base_label = infer_base_label(folder_name)

    if base_label is None:
        return None

    reward_mode = infer_reward_mode(folder_name)

    if reward_mode == "75":
        file_label = f"75_{base_label}"
    elif reward_mode == "full":
        file_label = base_label
    else:
        raise ValueError(f"Unknown reward_mode: {reward_mode!r}")

    return {
        "base_label": base_label,
        "reward_mode": reward_mode,
        "file_label": file_label,
    }


def check_experiment_folder_labels(source_root, verbose: bool = True):
    """
    Check whether each experiment folder can be mapped to:

        task label:
            first-pattern / first-position / pattern / position / couple

        reward mode:
            full / 75

    This function only checks folder naming.
    It does not inspect the MATLAB file contents.
    """
    source_root = Path(source_root)

    if not source_root.exists():
        raise FileNotFoundError(f"source_root does not exist: {source_root}")

    bad_folders = []
    good_folders = []

    for mouse_dir in sorted(source_root.iterdir()):
        if not mouse_dir.is_dir():
            continue

        for exp_dir in sorted(mouse_dir.iterdir()):
            if not exp_dir.is_dir():
                continue

            labels = infer_experiment_labels(exp_dir.name)

            if labels is None:
                bad_folders.append(exp_dir)
            else:
                good_folders.append((exp_dir, labels))

    if verbose:
        print(f"[Good folders] n={len(good_folders)}")

        for exp_dir, labels in good_folders:
            print(
                f"[OK] "
                f"task={labels['base_label']:14s} | "
                f"reward={labels['reward_mode']:4s} | "
                f"file_label={labels['file_label']:18s} | "
                f"{exp_dir}"
            )

        print(f"\n[Bad folders] n={len(bad_folders)}")

        for exp_dir in bad_folders:
            print(f"[Bad] {exp_dir}")

    if len(bad_folders) > 0:
        raise ValueError(
            "Some experiment folders cannot be assigned to a valid "
            "task label / reward mode. Please rename them before "
            "collecting neuro_type*.mat files."
        )

    return good_folders


def is_cutoff0_folder(folder_name: str) -> bool:
    """
    Only accept cut off 0 folders.

    Supported names
    ---------------
    cut off 0
    cut_off_0
    cut-off-0
    cutoff0
    """
    name = folder_name.lower()
    name = name.replace("_", " ")
    name = name.replace("-", " ")
    name = " ".join(name.split())

    if name == "cut off 0":
        return True

    if name == "cutoff0":
        return True

    return False


def collect_neuro_type_mat_files(
    source_root,
    data_root,
    overwrite: bool = False,
    verbose: bool = True,
):
    """
    collect neuro_type*.mat files from raw 2P data structure.
    

    Rules
    --------
    Full reward:
        neuro_type_xxx_pattern.mat
        neuro_type_xxx_position.mat
        neuro_type_xxx_couple.mat

    75% reward:
        neuro_type_xxx_75_pattern.mat
        neuro_type_xxx_75_position.mat
        neuro_type_xxx_75_couple.mat

    priority
    ------
    1. If exp_dir contains a cut off 0 folder:
        Only read neuro_type*.mat files under cut off 0.
        
    2. If exp_dir does not contain a cut off 0 folder:
        Read neuro_type*.mat files under exp_dir.
        
    3. If neither cut off 0 nor exp_dir contains neuro_type*.mat:
        Give a warning and skip.

    Parameters
    ----------
    source_root : str or Path

    data_root : str or Path

    overwrite : bool
        whether to overwrite existing files in data_root.

    verbose : bool
        whether to print progress information.

    Returns
    -------
    copied_files : list[tuple[Path, Path]]
    """
    source_root = Path(source_root)
    data_root = Path(data_root)

    if not source_root.exists():
        raise FileNotFoundError(f"source_root does not exist: {source_root}")

    data_root.mkdir(parents=True, exist_ok=True)

    copied_files = []

    for mouse_dir in sorted(source_root.iterdir()):
        if not mouse_dir.is_dir():
            continue

        mouse_name = mouse_dir.name
        out_mouse_dir = data_root / mouse_name
        out_mouse_dir.mkdir(parents=True, exist_ok=True)

        for exp_dir in sorted(mouse_dir.iterdir()):
            if not exp_dir.is_dir():
                continue

            labels = infer_experiment_labels(exp_dir.name)

            if labels is None:
                if verbose:
                    print(f"[Skip exp] Cannot infer label: {exp_dir}")
                continue

            base_label = labels["base_label"]
            reward_mode = labels["reward_mode"]
            file_label = labels["file_label"]

            cutoff0_dirs = [
                d for d in sorted(exp_dir.iterdir())
                if d.is_dir() and is_cutoff0_folder(d.name)
            ]

            if len(cutoff0_dirs) > 0:
                search_dirs = cutoff0_dirs

                if verbose:
                    print(
                        f"[Use cut off 0] "
                        f"task={base_label}, reward={reward_mode} | "
                        f"{exp_dir}"
                    )

            else:
                search_dirs = [exp_dir]

                if verbose:
                    print(
                        f"[No cut off 0] "
                        f"task={base_label}, reward={reward_mode} | "
                        f"Use current exp dir instead: {exp_dir}"
                    )

            found_any_mat = False

            for search_dir in search_dirs:
                mat_files = sorted(search_dir.glob("neuro_type*.mat"))

                if len(mat_files) == 0:
                    if verbose:
                        print(f"[No mat] {search_dir}")
                    continue

                found_any_mat = True

                for mat_file in mat_files:
                    new_name = (
                        f"{mat_file.stem}_{file_label}{mat_file.suffix}"
                    )

                    dst_file = out_mouse_dir / new_name

                    if dst_file.exists() and not overwrite:
                        if verbose:
                            print(f"[Exists, skip] {dst_file}")
                        continue

                    shutil.copy2(mat_file, dst_file)
                    copied_files.append((mat_file, dst_file))

                    if verbose:
                        print(
                            f"[Copied] "
                            f"task={base_label}, reward={reward_mode} | "
                            f"{mat_file} -> {dst_file}"
                        )

            if not found_any_mat and verbose:
                print(
                    f"[Skip exp] No neuro_type*.mat found in cut off 0 "
                    f"or current exp dir: {exp_dir}"
                )

    return copied_files

In [18]:
source_root = r"/Backup1/LWX/Processed 2P/"
data_root = r"../../../data/HPC_2p/raw"

check_experiment_folder_labels(
    source_root=source_root,
    verbose=True,
)

copied = collect_neuro_type_mat_files(
    source_root=source_root,
    data_root=data_root,
    overwrite=False,
    verbose=True,
)

print(f"Copied {len(copied)} files.")

[Good folders] n=58
[OK] task=couple         | reward=full | file_label=couple             | /Backup1/LWX/Processed 2P/HP01/241211 coupled
[OK] task=first-pattern  | reward=full | file_label=first-pattern      | /Backup1/LWX/Processed 2P/HP01/241213 first-pattern
[OK] task=pattern        | reward=full | file_label=pattern            | /Backup1/LWX/Processed 2P/HP01/241214 pattern
[OK] task=position       | reward=full | file_label=position           | /Backup1/LWX/Processed 2P/HP01/241224 position
[OK] task=couple         | reward=full | file_label=couple             | /Backup1/LWX/Processed 2P/HP02/241220 coupled
[OK] task=first-position | reward=full | file_label=first-position     | /Backup1/LWX/Processed 2P/HP02/241221 first position
[OK] task=position       | reward=full | file_label=position           | /Backup1/LWX/Processed 2P/HP02/241222_position
[OK] task=position       | reward=full | file_label=position           | /Backup1/LWX/Processed 2P/HP02/241223_position
[OK] task=pa